# ECG Classification - PyTorch Transformer (v5-pytorch)✅ PyTorch Implementation✅ Fixed Data Leakage✅ Stronger Regularization✅ Direct ONNX Export

## STEP 1: Imports

In [ ]:
import pandas as pdimport numpy as npimport torchimport torch.nn as nnimport torch.optim as optimfrom torch.utils.data import Dataset, DataLoaderfrom sklearn.preprocessing import StandardScalerfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import classification_reportimport warningswarnings.filterwarnings('ignore')device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f'Device: {device}')RANDOM_STATE = 42torch.manual_seed(RANDOM_STATE)np.random.seed(RANDOM_STATE)

## STEP 2: Load Data

In [ ]:
df = pd.read_csv('../../dataset_aritmia_NEW.csv')print(f'Shape: {df.shape}')print(f'\nLabels:\n{df["label"].value_counts()}')

## STEP 3: Preprocessing - **FIXED DATA LEAKAGE**

In [ ]:
X = df.drop('label', axis=1).valuesy = df['label'].values# Split FIRSTX_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp)# Normalize - fit only on trainingscaler = StandardScaler()X_train_norm = scaler.fit_transform(X_train)X_val_norm = scaler.transform(X_val)X_test_norm = scaler.transform(X_test)# Reshape for TransformerX_train_r = X_train_norm.reshape(-1, 188, 1)X_val_r = X_val_norm.reshape(-1, 188, 1)X_test_r = X_test_norm.reshape(-1, 188, 1)print(f'Train: {X_train_r.shape}, Val: {X_val_r.shape}, Test: {X_test_r.shape}')

## STEP 4: Dataset

In [ ]:
class ECGDataset(Dataset):    def __init__(self, X, y):        self.X = torch.FloatTensor(X)        self.y = torch.LongTensor(y)    def __len__(self): return len(self.X)    def __getitem__(self, idx): return self.X[idx], self.y[idx]train_ds = ECGDataset(X_train_r, y_train)val_ds = ECGDataset(X_val_r, y_val)test_ds = ECGDataset(X_test_r, y_test)BATCH_SIZE = 32train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)print(f'Loaders: {len(train_loader)} batches')

## STEP 5: Transformer Model**Regularization:** dropout=0.5

In [ ]:
class PositionalEncoding(nn.Module):    def __init__(self, d_model, max_len=200):        super().__init__()        self.encoding = nn.Parameter(torch.randn(max_len, d_model), requires_grad=True)    def forward(self, x):        return x + self.encoding[:x.size(1), :]class ECG_Transformer(nn.Module):    def __init__(self, d_model=64, nhead=4, num_layers=3, num_classes=2, dropout=0.5):        super().__init__()        self.input_proj = nn.Linear(1, d_model)        self.pos_encoding = PositionalEncoding(d_model)        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=256, dropout=dropout, batch_first=True)        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)        self.global_pool = nn.AdaptiveAvgPool1d(1)        self.fc = nn.Sequential(            nn.Linear(d_model, 64),            nn.ReLU(),            nn.Dropout(dropout),            nn.Linear(64, num_classes)        )        def forward(self, x):        x = self.input_proj(x)        x = self.pos_encoding(x)        x = self.transformer(x)        x = x.permute(0, 2, 1)        x = self.global_pool(x).squeeze(-1)        return self.fc(x)model = ECG_Transformer(dropout=0.5).to(device)print(f'Parameters: {sum(p.numel() for p in model.parameters())}')

## STEP 6: Training Config

In [ ]:
from sklearn.utils.class_weight import compute_class_weightclass_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)class_weights = torch.FloatTensor(class_weights).to(device)criterion = nn.CrossEntropyLoss(weight=class_weights)optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=5)print('Config ready')

## STEP 7: Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):    model.train()    loss_sum, correct, total = 0, 0, 0    for X, y in loader:        X, y = X.to(device), y.to(device)        optimizer.zero_grad()        out = model(X)        loss = criterion(out, y)        loss.backward()        optimizer.step()        loss_sum += loss.item()        _, pred = out.max(1)        total += y.size(0)        correct += pred.eq(y).sum().item()    return loss_sum/len(loader), correct/totaldef val_epoch(model, loader, criterion, device):    model.eval()    loss_sum, correct, total = 0, 0, 0    with torch.no_grad():        for X, y in loader:            X, y = X.to(device), y.to(device)            out = model(X)            loss = criterion(out, y)            loss_sum += loss.item()            _, pred = out.max(1)            total += y.size(0)            correct += pred.eq(y).sum().item()    return loss_sum/len(loader), correct/total

## STEP 8: Training Loop

In [ ]:
EPOCHS, PATIENCE = 100, 10best_loss, counter = float('inf'), 0print('Training...')for e in range(EPOCHS):    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)    val_loss, val_acc = val_epoch(model, val_loader, criterion, device)    scheduler.step(val_loss)        if (e+1) % 5 == 0 or e < 5:        print(f'Epoch {e+1}: Train={tr_acc:.4f} Val={val_acc:.4f}')        if val_loss < best_loss:        best_loss = val_loss        counter = 0        torch.save({'model': model.state_dict(), 'epoch': e, 'val_acc': val_acc}, 'ecg_transformer_pytorch_best.pth')    else:        counter += 1        if counter >= PATIENCE:            print(f'Early stop at epoch {e+1}')            breakcp = torch.load('ecg_transformer_pytorch_best.pth')model.load_state_dict(cp['model'])print(f'\nBest: epoch {cp["epoch"]+1}, val_acc={cp["val_acc"]:.4f}')

## STEP 9: Evaluation

In [ ]:
test_loss, test_acc = val_epoch(model, test_loader, criterion, device)print(f'Test Accuracy: {test_acc:.4f}')model.eval()preds, labels = [], []with torch.no_grad():    for X, y in test_loader:        out = model(X.to(device))        _, p = out.max(1)        preds.extend(p.cpu().numpy())        labels.extend(y.numpy())print('\nClassification Report:')print(classification_report(labels, preds, target_names=['Normal', 'Abnormal']))

## STEP 10: Save & Export

In [ ]:
torch.save({'model': model.state_dict(), 'test_acc': test_acc}, 'ecg_transformer_v5_pytorch_final.pth')import joblibjoblib.dump(scaler, 'scaler_v5_pytorch.pkl')print('Models saved')try:    model.eval()    dummy = torch.randn(1, 188, 1).to(device)    torch.onnx.export(model, dummy, 'ecg_transformer_v5_pytorch_final.onnx',                     export_params=True, opset_version=13,                     input_names=['input'], output_names=['output'],                     dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}})    print('✓ ONNX exported: ecg_transformer_v5_pytorch_final.onnx')    import onnxruntime as ort    ort.InferenceSession('ecg_transformer_v5_pytorch_final.onnx')    print('✓ ONNX verified')except Exception as e:    print(f'ONNX export failed: {e}')